In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Load data
df = pd.read_csv("cleaned_single_family_sales.csv", low_memory=False)
print("Original shape:", df.shape)

Original shape: (320506, 83)


In [3]:
# Select useful columns
keep_cols = [
    "CloseDate",
    "ClosePrice",             # target

    # numeric
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "YearBuilt",
    "DaysOnMarket",
    "ParkingTotal",

    # categorical
    "PropertySubType",
    "CountyOrParish",
    "City",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN"
]
df = df[keep_cols].copy()

In [4]:
# Rename columns
df = df.rename(columns={
    "BedroomsTotal": "Bedrooms",
    "BathroomsTotalInteger": "Bathrooms",
    "LotSizeSquareFeet": "LotSize"
})

In [5]:
# Convert data types
numeric_cols = [
     "ClosePrice",
    "LivingArea",
    "Bedrooms",
    "Bathrooms",
    "LotSize",
    "GarageSpaces",
    "YearBuilt",
    "DaysOnMarket",
    "ParkingTotal"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

In [6]:
# Missing values
# Numeric → median
for col in numeric_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )
# Categorical → Unknown
cat_cols = [
    "PropertySubType",
    "CountyOrParish",
    "City",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN"
]
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

In [7]:
# Encode categorical features
df = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True
)
print("After encoding:", df.shape)

After encoding: (320506, 1213)


In [8]:
# Normalize numeric variables
scale_cols = [
    "LivingArea",
    "Bedrooms",
    "Bathrooms",
    "LotSize",
    "GarageSpaces",
    "YearBuilt"
]
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [9]:
# Time-based train/test split
# Test = most recent month
# Train = previous X months
latest_month = (df["CloseDate"].dt.to_period("M").max())
test = df[df["CloseDate"].dt.to_period("M") == latest_month]
X_MONTHS = 12
train_start = (
    latest_month - X_MONTHS
)
train = df[
    (
        df["CloseDate"]
        .dt.to_period("M")
        >= train_start
    )
    &
    (
        df["CloseDate"]
        .dt.to_period("M")
        < latest_month
    )
]
print("Train:", train.shape)
print("Test:", test.shape)

Train: (129973, 1213)
Test: (12024, 1213)


In [10]:
# Save outputs
train.to_csv("train_processed.csv", index=False)
test.to_csv("test_processed.csv", index=False)
df.to_csv("cleaned_preprocessed.csv", index=False)
print("Finished preprocessing")

Finished preprocessing
